# Phase 4: LLMs & Prompt Engineering
## Day 20: InformationExtractionProject

Date: 2026-04-24

### Learning objectives
- Build an end-to-end information extraction pipeline.
- Convert campaign summaries into structured JSON.
- Validate extracted fields.
- Add retry and repair logic.
- Analyze the final structured dataset.

In [ ]:
import json
import re
import textwrap
from typing import Optional, Dict, Any, List
from pprint import pprint

import pandas as pd

try:
    from pydantic import BaseModel, Field, ValidationError
    PYDANTIC_AVAILABLE = True
except Exception:
    BaseModel = object
    Field = None
    ValidationError = Exception
    PYDANTIC_AVAILABLE = False
    print("Pydantic is not installed. Fallback validation will be used.")

def show(title, content):
    print("\n" + "=" * 76)
    print(title)
    print("=" * 76)
    print(textwrap.dedent(str(content)).strip())

def model_to_dict(model):
    if hasattr(model, "model_dump"):
        return model.model_dump()
    if hasattr(model, "dict"):
        return model.dict()
    return model

print("Setup complete.")
print("Pydantic available:", PYDANTIC_AVAILABLE)

In [ ]:
campaign_reports = [
    {
        "report_id": "R001",
        "summary": '''
        Campaign: Spring Coffee Push
        Channel: Instagram
        Region: Berlin
        Spend: 1200 EUR
        Impressions: 45000
        Clicks: 3420
        Conversions: 184
        Sentiment: positive
        Notes: Strong CTR. Conversions improved after adding a limited-time discount.
        '''
    },
    {
        "report_id": "R002",
        "summary": '''
        Bank App Onboarding ran on Email in Germany.
        It spent 800 EUR, got 980 clicks, and generated 42 conversions from 18000 impressions.
        Sentiment was neutral. The subject line may be too generic.
        '''
    },
    {
        "report_id": "R003",
        "summary": '''
        Yoga Studio Trial used TikTok for a Berlin audience.
        Budget was 650 EUR. It produced 2100 clicks, 165 conversions, and 27000 impressions.
        Sentiment: positive. Short beginner-friendly videos performed well.
        '''
    },
    {
        "report_id": "R004",
        "summary": '''
        Campaign: Premium Card Upgrade
        Channel: LinkedIn
        Region: Germany
        Spend: 1500 EUR
        Impressions: 22000
        Clicks: 740
        Conversions: 31
        Sentiment: negative
        Notes: Cost per conversion is high. Audience may be too broad.
        '''
    },
    {
        "report_id": "R005",
        "summary": '''
        Summer Bagel Launch ran on Instagram in Berlin.
        The campaign spent 900 EUR and received 39000 impressions.
        It got 2800 clicks and 220 conversions. Sentiment was positive.
        Food photos and short reels performed best.
        '''
    }
]

show("First raw campaign report", campaign_reports[0]["summary"])
print("Number of reports:", len(campaign_reports))

## 1. Project goal

The goal is to turn messy campaign text into clean JSON records.

This is common in real AI engineering work. You receive reports, emails, PDFs, tickets, or summaries. Your pipeline extracts fields and makes the data usable.

In [ ]:
target_schema = {
    "report_id": "string",
    "campaign": "string",
    "channel": "string or null",
    "region": "string or null",
    "spend_eur": "number or null",
    "impressions": "integer or null",
    "clicks": "integer or null",
    "conversions": "integer or null",
    "sentiment": "positive | neutral | negative | null",
    "notes": "string or null"
}

print(json.dumps(target_schema, indent=2))

## 2. Build the extraction prompt

A strong extraction prompt has a clear task, raw text, schema, and strict rules.

For production pipelines, keep the output format boring and predictable.

In [ ]:
def build_extraction_prompt(report_id, summary, schema):
    return f'''
Task:
Extract campaign information from the report.

Report ID:
{report_id}

Raw report:
{summary.strip()}

Rules:
- Return valid JSON only.
- Do not include markdown.
- Do not include explanations.
- Use null when a field is missing.
- Use numbers for numeric fields.
- Sentiment must be positive, neutral, negative, or null.

JSON schema:
{json.dumps(schema, indent=2)}
'''.strip()

prompt = build_extraction_prompt(
    campaign_reports[0]["report_id"],
    campaign_reports[0]["summary"],
    target_schema
)

show("Extraction prompt", prompt)

## 3. Create a local mock extractor

In a real project, this step would call an LLM.

Here, we use regex rules as a local mock. This keeps the notebook runnable without API keys or Ollama.

In [ ]:
def find_first(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        match = re.search(pattern, text, flags=flags)
        if match:
            return match.group(1).strip()
    return None

def find_number(patterns, text):
    value = find_first(patterns, text)
    return int(value) if value is not None else None

def normalize_channel(text):
    for channel in ["Instagram", "Email", "TikTok", "LinkedIn", "Google"]:
        if channel.lower() in text.lower():
            return channel
    return None

def normalize_region(text):
    for region in ["Berlin", "Germany", "Turkey", "Spain"]:
        if region.lower() in text.lower():
            return region
    return None

def normalize_sentiment(text):
    text_lower = text.lower()
    for sentiment in ["positive", "neutral", "negative"]:
        if sentiment in text_lower:
            return sentiment
    return None

def mock_llm_extract(report):
    text = " ".join(report["summary"].split())

    campaign = find_first([
        r"Campaign:\s*([A-Za-z\s]+?)\s+Channel:",
        r"([A-Za-z\s]+?)\s+ran on",
        r"([A-Za-z\s]+?)\s+used\s+TikTok",
    ], text)

    spend = find_number([r"Spend:\s*(\d+)\s*EUR", r"spent\s+(\d+)\s*EUR", r"Budget was\s+(\d+)\s*EUR"], text)
    impressions = find_number([r"Impressions:\s*(\d+)", r"(\d+)\s+impressions", r"from\s+(\d+)\s+impressions"], text)
    clicks = find_number([r"Clicks:\s*(\d+)", r"got\s+(\d+)\s+clicks", r"produced\s+(\d+)\s+clicks"], text)
    conversions = find_number([r"Conversions:\s*(\d+)", r"generated\s+(\d+)\s+conversions", r"(\d+)\s+conversions"], text)

    notes = find_first([r"Notes:\s*(.+)", r"Sentiment was \w+\.\s*(.+)", r"Sentiment:\s*\w+\.\s*(.+)"], text)

    return {
        "report_id": report["report_id"],
        "campaign": campaign,
        "channel": normalize_channel(text),
        "region": normalize_region(text),
        "spend_eur": spend,
        "impressions": impressions,
        "clicks": clicks,
        "conversions": conversions,
        "sentiment": normalize_sentiment(text),
        "notes": notes
    }

mock_output = mock_llm_extract(campaign_reports[0])
pprint(mock_output)

In [ ]:
for report in campaign_reports:
    print("\nReport:", report["report_id"])
    pprint(mock_llm_extract(report))

## 4. Parse JSON safely

LLM output arrives as text.

Your first job is to parse it safely and handle broken JSON without crashing the whole pipeline.

In [ ]:
def safe_json_loads(text):
    try:
        return json.loads(text), None
    except json.JSONDecodeError as error:
        return None, str(error)

good_json = json.dumps(mock_llm_extract(campaign_reports[0]))
bad_json = good_json[:-1] + ",}"

for item in [good_json, bad_json]:
    data, error = safe_json_loads(item)
    print("\nInput preview:", item[:80] + "...")
    print("Parsed:", data is not None)
    print("Error:", error)

In [ ]:
def extract_json_block(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    return match.group(0) if match else None

def repair_json_text(text):
    repaired = text.strip()
    repaired = repaired.replace("```json", "").replace("```", "").strip()
    repaired = re.sub(r",\s*}", "}", repaired)
    repaired = re.sub(r",\s*]", "]", repaired)
    return repaired

messy_llm_output = '''
Here is the result:
```json
{
  "report_id": "R001",
  "campaign": "Spring Coffee Push",
  "channel": "Instagram",
  "region": "Berlin",
  "spend_eur": 1200,
  "impressions": 45000,
  "clicks": 3420,
  "conversions": 184,
  "sentiment": "positive",
  "notes": "Strong CTR.",
}
```
'''

json_block = extract_json_block(messy_llm_output)
repaired = repair_json_text(json_block)
data, error = safe_json_loads(repaired)

pprint(data)
print("Error:", error)

## 5. Validate the extracted record

Validation protects your pipeline from bad data.

It checks field names, types, allowed values, and simple numeric rules.

In [ ]:
if PYDANTIC_AVAILABLE:
    class CampaignRecord(BaseModel):
        report_id: str
        campaign: str
        channel: Optional[str] = None
        region: Optional[str] = None
        spend_eur: Optional[float] = Field(default=None, ge=0)
        impressions: Optional[int] = Field(default=None, ge=0)
        clicks: Optional[int] = Field(default=None, ge=0)
        conversions: Optional[int] = Field(default=None, ge=0)
        sentiment: Optional[str] = None
        notes: Optional[str] = None
else:
    CampaignRecord = None

allowed_sentiments = {"positive", "neutral", "negative", None}

def fallback_validate_campaign(data):
    required = [
        "report_id", "campaign", "channel", "region", "spend_eur",
        "impressions", "clicks", "conversions", "sentiment", "notes"
    ]
    errors = []

    for field in required:
        if field not in data:
            errors.append(f"Missing field: {field}")

    if not isinstance(data.get("report_id"), str):
        errors.append("report_id must be a string")
    if not isinstance(data.get("campaign"), str) or not data.get("campaign"):
        errors.append("campaign must be a non-empty string")

    for field in ["spend_eur", "impressions", "clicks", "conversions"]:
        value = data.get(field)
        if value is not None and not isinstance(value, (int, float)):
            errors.append(f"{field} must be numeric or null")
        if isinstance(value, (int, float)) and value < 0:
            errors.append(f"{field} must be non-negative")

    if data.get("sentiment") not in allowed_sentiments:
        errors.append("sentiment must be positive, neutral, negative, or null")

    if errors:
        raise ValueError(errors)

    return data

def validate_campaign(data):
    if PYDANTIC_AVAILABLE:
        model = CampaignRecord(**data)
        model_data = model_to_dict(model)
        if model_data.get("sentiment") not in allowed_sentiments:
            raise ValueError("sentiment must be positive, neutral, negative, or null")
        return model
    return fallback_validate_campaign(data)

validated = validate_campaign(mock_llm_extract(campaign_reports[0]))
pprint(model_to_dict(validated))

In [ ]:
invalid_record = {
    "report_id": "R999",
    "campaign": "",
    "channel": "Email",
    "region": "Germany",
    "spend_eur": -100,
    "impressions": 1000,
    "clicks": "many",
    "conversions": 20,
    "sentiment": "happy",
    "notes": None
}

try:
    validate_campaign(invalid_record)
except Exception as error:
    print("Validation failed:")
    print(error)

## 6. Add retry logic

Retry logic helps when the first extraction is broken.

The retry prompt should include the original text, the bad output, the error, and the schema.

In [ ]:
def make_retry_prompt(report, bad_output, error_message, schema):
    return f'''
The previous extraction failed.

Report ID:
{report["report_id"]}

Original report:
{report["summary"].strip()}

Bad output:
{bad_output}

Error:
{error_message}

Return a corrected result.

Rules:
- Return valid JSON only.
- Do not include markdown.
- Do not include explanations.
- Use this exact schema:
{json.dumps(schema, indent=2)}
'''.strip()

bad_output = '{"report_id": "R002", "campaign": "Bank App Onboarding", "clicks": "many",}'
_, error = safe_json_loads(bad_output)

retry_prompt = make_retry_prompt(campaign_reports[1], bad_output, error, target_schema)
show("Retry prompt", retry_prompt)

In [ ]:
def mock_retry_response(report):
    # In production, this would call the LLM again with make_retry_prompt.
    clean_record = mock_llm_extract(report)
    return json.dumps(clean_record, indent=2)

retry_response = mock_retry_response(campaign_reports[1])
print(retry_response)

data, error = safe_json_loads(retry_response)
validated = validate_campaign(data)

print("\nValidated retry response:")
pprint(model_to_dict(validated))

## 7. Build the full pipeline

Now we combine prompt creation, mock LLM extraction, JSON parsing, repair, validation, and retry.

The output includes both the clean record and useful metadata.

In [ ]:
def parse_and_validate_output(output_text):
    json_text = extract_json_block(output_text) or output_text
    data, parse_error = safe_json_loads(json_text)

    if parse_error:
        repaired = repair_json_text(json_text)
        data, parse_error = safe_json_loads(repaired)

    if parse_error:
        return None, f"Parse error: {parse_error}"

    try:
        validated = validate_campaign(data)
        return model_to_dict(validated), None
    except Exception as validation_error:
        return None, f"Validation error: {validation_error}"

def run_extraction_pipeline(report, simulate_broken=False):
    prompt = build_extraction_prompt(report["report_id"], report["summary"], target_schema)

    first_record = mock_llm_extract(report)
    first_output = json.dumps(first_record, indent=2)

    if simulate_broken:
        first_output = first_output[:-2] + ",\n}"

    record, error = parse_and_validate_output(first_output)
    retry_used = False

    if error:
        retry_used = True
        retry_output = mock_retry_response(report)
        record, error = parse_and_validate_output(retry_output)
    else:
        retry_output = None

    return {
        "report_id": report["report_id"],
        "prompt_preview": prompt[:180] + "...",
        "first_output": first_output,
        "retry_used": retry_used,
        "retry_output": retry_output,
        "record": record,
        "error": error
    }

single_result = run_extraction_pipeline(campaign_reports[0])
pprint(single_result)

In [ ]:
pipeline_results = []

for index, report in enumerate(campaign_reports):
    result = run_extraction_pipeline(report, simulate_broken=(index == 2))
    pipeline_results.append(result)

for result in pipeline_results:
    print("\nReport:", result["report_id"])
    print("Retry used:", result["retry_used"])
    print("Error:", result["error"])
    pprint(result["record"])

## 8. Create the final dataset

After extraction, we can use normal data analysis tools.

This is why structured output is powerful.

In [ ]:
records = [item["record"] for item in pipeline_results if item["error"] is None]
df = pd.DataFrame(records)

df["ctr"] = df["clicks"] / df["impressions"]
df["conversion_rate"] = df["conversions"] / df["clicks"]
df["cost_per_conversion"] = df["spend_eur"] / df["conversions"]

df

In [ ]:
summary_by_channel = (
    df.groupby("channel", as_index=False)
      .agg(
          total_spend=("spend_eur", "sum"),
          total_clicks=("clicks", "sum"),
          total_conversions=("conversions", "sum"),
          avg_conversion_rate=("conversion_rate", "mean")
      )
      .sort_values("total_conversions", ascending=False)
)

summary_by_channel

In [ ]:
best_campaign = df.sort_values("conversion_rate", ascending=False).iloc[0]
worst_cpc_campaign = df.sort_values("cost_per_conversion", ascending=False).iloc[0]

print("Best conversion rate:")
print(best_campaign[["campaign", "channel", "conversion_rate"]])

print("\nHighest cost per conversion:")
print(worst_cpc_campaign[["campaign", "channel", "cost_per_conversion"]])

## 9. Quality checks

A production pipeline should check the final data before using it.

Simple checks can catch many problems early.

In [ ]:
def quality_checks(df):
    checks = {}
    checks["no_missing_campaign"] = df["campaign"].notna().all()
    checks["non_negative_spend"] = (df["spend_eur"] >= 0).all()
    checks["clicks_not_above_impressions"] = (df["clicks"] <= df["impressions"]).all()
    checks["conversions_not_above_clicks"] = (df["conversions"] <= df["clicks"]).all()
    checks["valid_sentiment"] = df["sentiment"].isin(["positive", "neutral", "negative"]).all()
    return checks

checks = quality_checks(df)
pprint(checks)

assert all(checks.values())
print("All quality checks passed.")

In [ ]:
def make_project_report(df, pipeline_results):
    total_reports = len(pipeline_results)
    successful = sum(item["error"] is None for item in pipeline_results)
    retries = sum(item["retry_used"] for item in pipeline_results)

    return {
        "total_reports": total_reports,
        "successful_extractions": successful,
        "failed_extractions": total_reports - successful,
        "retry_count": retries,
        "total_spend_eur": round(df["spend_eur"].sum(), 2),
        "total_conversions": int(df["conversions"].sum()),
        "best_campaign_by_conversion_rate": df.sort_values("conversion_rate", ascending=False).iloc[0]["campaign"]
    }

project_report = make_project_report(df, pipeline_results)
pprint(project_report)

## 10. Optional real LLM integration shape

The notebook uses a mock extractor.

In a real app, replace `mock_llm_extract` with a function that sends the prompt to OpenAI, Ollama, or another model.

In [ ]:
def real_llm_function_shape(prompt):
    '''
    This is only a template.

    For OpenAI:
    - send the prompt to a chat completion endpoint
    - request JSON-only output
    - return response text

    For Ollama:
    - POST to http://localhost:11434/api/generate
    - pass model and prompt
    - set stream to False
    - return response["response"]
    '''
    raise NotImplementedError("Replace this with your real LLM call.")

print("Real LLM integration shape is defined as a template.")

## Tricky bits

Information extraction projects fail when the pipeline trusts the model too much.

Always parse, validate, repair small formatting issues, retry when needed, and run quality checks.

In [ ]:
broken_cases = [
    '{"report_id": "R001", "campaign": "Demo", "clicks": 100,}',
    'Here is the JSON: {"report_id": "R001", "campaign": "Demo", "clicks": 100}',
    '{"report_id": "R001", "campaign": "Demo", "sentiment": "excited"}',
]

for item in broken_cases:
    print("\nCase:", item)
    record, error = parse_and_validate_output(item)
    print("Record:", record)
    print("Error:", error)

In [ ]:
# Mistake: using parsed JSON without checking required fields.

partial_record = {"report_id": "R777", "campaign": "Partial Campaign"}

try:
    validate_campaign(partial_record)
except Exception as error:
    print("This failed because the schema is incomplete:")
    print(error)

## Trick questions

1. Why not store raw LLM text directly in a database?

<details>
<summary>Answer</summary>

Raw text is hard to filter, aggregate, validate, and join with other data. Structured fields are easier to use.

</details>

2. Is regex enough for all extraction tasks?

<details>
<summary>Answer</summary>

No. Regex works for predictable text. LLMs help when the wording is flexible or messy.

</details>

3. Why should retry prompts include the validation error?

<details>
<summary>Answer</summary>

The error tells the model exactly what needs to be fixed.

</details>

4. What should you do after all records are extracted?

<details>
<summary>Answer</summary>

Run quality checks before using the data for analysis or decisions.

</details>

5. What is the best output format for a data pipeline?

<details>
<summary>Answer</summary>

A strict structured format such as JSON that follows a known schema.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Build an extraction prompt for report R001.

report = campaign_reports[0]

my_prompt = ___

assert isinstance(my_prompt, str)
assert "Return valid JSON only" in my_prompt
assert report["report_id"] in my_prompt
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Extract one campaign record with the mock extractor.

record = ___

assert isinstance(record, dict)
assert record["report_id"] == "R002"
assert record["campaign"] == "Bank App Onboarding"
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Convert a record to JSON text.

record = mock_llm_extract(campaign_reports[2])
json_text = ___

assert isinstance(json_text, str)
assert "Yoga Studio Trial" in json_text
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Parse JSON text safely.

json_text = '{"report_id": "R999", "campaign": "Demo"}'
data, error = ___

assert data["report_id"] == "R999"
assert error is None
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Repair a JSON string with a trailing comma.

broken = '{"report_id": "R999", "campaign": "Demo",}'
repaired = ___
data = json.loads(repaired)

assert data["campaign"] == "Demo"
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Run the full pipeline on report R003.

result = ___

assert result["report_id"] == "R003"
assert result["error"] is None
assert result["record"]["campaign"] == "Yoga Studio Trial"
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Create a DataFrame from successful pipeline records.

records = [item["record"] for item in pipeline_results if item["error"] is None]
exercise_df = ___

assert len(exercise_df) == len(records)
assert "campaign" in exercise_df.columns
print("Exercise 7 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
my_prompt = build_extraction_prompt(
    report["report_id"],
    report["summary"],
    target_schema
)
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
record = mock_llm_extract(campaign_reports[1])
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
json_text = json.dumps(record, indent=2)
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
data, error = safe_json_loads(json_text)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
repaired = repair_json_text(broken)
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
result = run_extraction_pipeline(campaign_reports[2])
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
exercise_df = pd.DataFrame(records)
```

</details>

## Cumulative review exercises

These mix topics from Days 10 to 19. Fill in `___` and run each cell.

In [ ]:
# Review 1: SHAP
# Complete the sentence.

shap_sentence = ___

assert "feature" in shap_sentence.lower()
assert "prediction" in shap_sentence.lower()
print("Review 1 passed.")

In [ ]:
# Review 2: Text preprocessing
# Lowercase and split a sentence into tokens.

sentence = "The Campaign Performed Well"
tokens = ___

assert tokens == ["the", "campaign", "performed", "well"]
print("Review 2 passed.")

In [ ]:
# Review 3: Transformers
# Fill the missing word.

attention_sentence = "Attention helps a model focus on relevant ___."

missing_word = ___

assert missing_word == "tokens"
print("Review 3 passed.")

In [ ]:
# Review 4: Hugging Face
# Name the quick task helper.

hf_helper = ___

assert hf_helper == "pipeline"
print("Review 4 passed.")

In [ ]:
# Review 5: Fine-tuning
# Pick the common Hugging Face training class.

training_class = ___

assert training_class == "Trainer"
print("Review 5 passed.")

In [ ]:
# Review 6: Complaint classification
# Create a prediction label.

prediction_label = ___

assert prediction_label in ["billing", "delivery", "technical", "other"]
print("Review 6 passed.")

In [ ]:
# Review 7: OpenAI API roles
# Fill the standard chat roles.

roles = ___

assert roles == ["system", "user", "assistant"]
print("Review 7 passed.")

In [ ]:
# Review 8: Ollama
# Fill the default local generate endpoint.

ollama_generate_url = ___

assert ollama_generate_url == "http://localhost:11434/api/generate"
print("Review 8 passed.")

In [ ]:
# Review 9: Prompt engineering
# Choose the prompting style that includes examples.

style = ___

assert style.lower() == "few-shot"
print("Review 9 passed.")

In [ ]:
# Review 10: Structured output
# Parse JSON text.

text = '{"campaign": "Demo", "clicks": 100}'
parsed = ___

assert parsed["clicks"] == 100
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
shap_sentence = "SHAP explains how each feature contributes to a prediction."

# Review 2
tokens = sentence.lower().split()

# Review 3
missing_word = "tokens"

# Review 4
hf_helper = "pipeline"

# Review 5
training_class = "Trainer"

# Review 6
prediction_label = "billing"

# Review 7
roles = ["system", "user", "assistant"]

# Review 8
ollama_generate_url = "http://localhost:11434/api/generate"

# Review 9
style = "few-shot"

# Review 10
parsed = json.loads(text)
```

</details>

In [ ]:
cheat_sheet = '''
DAY 20 CHEAT SHEET: INFORMATION EXTRACTION PROJECT

Pipeline steps:
1. Collect raw text.
2. Build a strict extraction prompt.
3. Ask for JSON only.
4. Parse JSON safely.
5. Repair simple formatting issues.
6. Validate fields and types.
7. Retry when parsing or validation fails.
8. Store clean records.
9. Run quality checks.
10. Analyze the final dataset.

Good schema fields:
- report_id
- campaign
- channel
- region
- spend_eur
- impressions
- clicks
- conversions
- sentiment
- notes

Useful checks:
- clicks <= impressions
- conversions <= clicks
- spend_eur >= 0
- sentiment is in allowed values
- campaign is not missing
'''

print(cheat_sheet)

## Next up: Day 21 — TesseractBasics

You will start Phase 5 and learn OCR basics with pytesseract, language packs, psm and oem modes, and bounding boxes.